# Cattle Breed Recognition Training in Colab

This notebook loads the dataset, applies preprocessing and augmentation, trains a MobileNetV2 classifier, and saves the trained model plus labels.

Use Google Drive if you want the model artifacts to persist after the Colab session ends.

In [ ]:
import os
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print(tf.__version__)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Update this if your dataset is stored in a different Drive folder.
PROJECT_ROOT = Path('/content/drive/MyDrive/cattle-breed-recognition-ai')
DATASET_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'models'
MODEL_SAVE_PATH = OUTPUT_DIR / 'cattle_breed_mobilenetv2.keras'
LABELS_SAVE_PATH = OUTPUT_DIR / 'labels.txt'

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 10
SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
if not DATASET_DIR.exists():
    raise FileNotFoundError(f'Dataset directory not found: {DATASET_DIR}')

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print('Classes:', class_names)

train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
class_counts = {}
for class_name in class_names:
    class_dir = DATASET_DIR / class_name
    class_counts[class_name] = len([path for path in class_dir.iterdir() if path.is_file()])

num_samples = sum(class_counts.values())
num_classes = len(class_names)

class_weights = {
    index: (num_samples / (num_classes * class_counts[class_name]))
    for index, class_name in enumerate(class_names)
    if class_counts[class_name] > 0
}

print('Class counts:', class_counts)
print('Class weights:', class_weights)

In [ ]:
augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ],
    name='augmentation',
)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = augmentation(inputs)
x = tf.keras.layers.Lambda(preprocess_input)(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)

FINE_TUNE_EPOCHS = 5
TOTAL_EPOCHS = EPOCHS + FINE_TUNE_EPOCHS

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TOTAL_EPOCHS,
    initial_epoch=history.epoch[-1] + 1,
    callbacks=callbacks,
    class_weight=class_weights,
)


In [ ]:
model.save(MODEL_SAVE_PATH)

with LABELS_SAVE_PATH.open('w', encoding='utf-8') as label_file:
    for name in class_names:
        label_file.write(f'{name}\n')

print(f'Fine-tuned model saved to: {MODEL_SAVE_PATH}')
print(f'Labels saved to: {LABELS_SAVE_PATH}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

loaded_model = tf.keras.models.load_model(MODEL_SAVE_PATH)

class_names = [line.strip() for line in LABELS_SAVE_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]

for images, labels in val_ds.take(1):
    predictions = loaded_model.predict(images, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)

    plt.figure(figsize=(15, 10))
    for i in range(min(9, images.shape[0])):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        true_label = class_names[int(labels[i])]
        predicted_label = class_names[int(predicted_classes[i])]
        confidence = float(np.max(predictions[i]))
        plt.title(f'True: {true_label}\nPred: {predicted_label} ({confidence:.2f})')
        plt.axis('off')
    plt.tight_layout()
    plt.show()

print('Test complete: model loaded successfully and predictions were generated.')